# 01 Stažení referenčního genomu (GRCh38 / hg38), úkol 1.1

Jednorázový notebook. Stáhne **UCSC hg38 analysis set** (bez alternativních kontigů,
repetice soft-maskované malými písmeny), ověří MD5, ponechá jen hlavní chromozomy,
vytvoří `.fai` index a uloží vše do `P.genome_dir` (v Colabu na Google Drive).

Stačí CPU runtime. Trvá zhruba 10–20 minut, na Drive zabere asi 4 GB.

## 1. Nastavení

In [ ]:
WINDOW = 1024   # jen kvůli jednotné úvodní buňce, tady se nepoužívá

# Hlavní server UCSC a evropské zrcadlo (stejné soubory); zkouší se v tomto pořadí
MIRRORS = [
    "https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/analysisSet",
    "https://hgdownload-euro.soe.ucsc.edu/goldenPath/hg38/bigZips/analysisSet",
]
FA_GZ = "hg38.analysisSet.fa.gz"
OUT_FA = "hg38.main.fa"

# chr1–22, X, Y a mitochondrie. Výběr pro trénink (1–22 + X) se dělá až při generování oken.
KEEP = [f"chr{i}" for i in range(1, 23)] + ["chrX", "chrY", "chrM"]

## Prostředí

Funguje v Colabu (prohlížeč i VS Code) i lokálně. V Colabu připojí Google Drive,
stáhne kód z GitHubu a zkopíruje genom na rychlý lokální disk. Jinde se cesty
řídí proměnnou `DNALM_ROOT`. Všechny cesty jsou v objektu `P` (`P.fasta`,
`P.windows("train")`, `P.tokenizer("bpe_600")`, `P.figures`, ...).

Ve VS Code s Colab rozšířením: pokud se Drive nepřipojí, použij příkaz
`Colab: Mount Google Drive to Server...` a buňku spusť znovu.

In [ ]:
# Úvodní buňka (stejná ve všech notebookách, neměnit).
# Colab: stáhne aktuální kód z GitHubu. Lokálně (Mac, školní GPU): použije nainstalované repo.
import importlib.util, sys
if importlib.util.find_spec("google.colab"):
    !git clone -q https://github.com/Valasik0/dna-lm.git /content/dna-lm 2>/dev/null || git -C /content/dna-lm pull -q
    !git -C /content/dna-lm log --oneline -1
    if "/content/dna-lm/src" not in sys.path:
        sys.path.insert(0, "/content/dna-lm/src")

from dnalm.utils.colab import setup
P = setup(window=WINDOW, genome=False)

## 2. Pomocné funkce

In [ ]:
import gzip
import hashlib
import json
import os
import shutil
import socket
import time
import urllib.request
from datetime import datetime, timezone

from tqdm.auto import tqdm


def check_dns(urls):
    """Vypíše, které servery jde přeložit na IP (diagnostika síťových chyb)."""
    for u in urls:
        host = u.split("/")[2]
        try:
            print(f"OK     {host} -> {socket.gethostbyname(host)}")
        except OSError as e:
            print(f"CHYBA  {host}: {e}")


def with_mirrors(func, mirrors, retries=3, wait=10):
    """Zavolá func(base_url) postupně pro každé zrcadlo, s několika pokusy. Vrací base_url."""
    last = None
    for base in mirrors:
        for attempt in range(1, retries + 1):
            try:
                print(f"[{base.split('/')[2]}] pokus {attempt}/{retries}")
                func(base)
                return base
            except OSError as e:  # URLError, DNS, timeout, přerušené spojení
                last = e
                print(f"   selhalo: {e}")
                time.sleep(wait)
    raise RuntimeError(f"Žádné zrcadlo nefunguje, poslední chyba: {last}")


def download(url, dest):
    """Stáhne soubor s progress barem; pokud už existuje celý, přeskočí."""
    with urllib.request.urlopen(url) as r:
        total = int(r.headers.get("Content-Length", 0))
        if os.path.exists(dest) and os.path.getsize(dest) == total:
            print(f"Už staženo: {dest}")
            return
        with open(dest, "wb") as f, tqdm(
            total=total, unit="B", unit_scale=True, desc=os.path.basename(dest)
        ) as bar:
            while chunk := r.read(1 << 20):
                f.write(chunk)
                bar.update(len(chunk))


def md5sum(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while chunk := f.read(1 << 24):
            h.update(chunk)
    return h.hexdigest()


def expected_md5(md5_url, filename):
    text = urllib.request.urlopen(md5_url).read().decode()
    for line in text.splitlines():
        parts = line.split()
        if len(parts) == 2 and parts[1].lstrip("*") == filename:
            return parts[0]
    raise ValueError(f"{filename} není v {md5_url}")


def filter_fasta(src_gz, dst_fa, keep):
    """
    Projde gzip FASTA jedním průchodem, ponechá jen chromozomy z `keep`,
    zapíše nekomprimovanou FASTA, vytvoří k ní .fai index a vrátí statistiky.
    """
    keep = set(keep)
    stats, fai = [], []
    cur = None
    offset = 0

    def close(c):
        if c is not None:
            stats.append({k: c[k] for k in ("chrom", "length", "n_count", "softmasked")})
            fai.append(f"{c['chrom']}\t{c['length']}\t{c['offset']}\t{c['lb']}\t{c['lb'] + 1}")

    with gzip.open(src_gz, "rt") as fin, open(dst_fa, "w") as fout:
        writing = False
        for line in tqdm(fin, desc="Filtrování", unit=" řádků", mininterval=5):
            if line.startswith(">"):
                close(cur)
                cur = None
                name = line[1:].split()[0]
                writing = name in keep
                if writing:
                    header = f">{name}\n"
                    fout.write(header)
                    offset += len(header)
                    cur = {"chrom": name, "length": 0, "n_count": 0, "softmasked": 0,
                           "offset": offset, "lb": None, "short_seen": False}
                continue
            if not writing:
                continue
            seq = line.rstrip("\n")
            n = len(seq)
            if cur["lb"] is None:
                cur["lb"] = n
            elif cur["short_seen"] or n > cur["lb"]:
                raise ValueError(f"Nestejná délka řádků v {cur['chrom']}, .fai by byl špatně")
            if n < cur["lb"]:
                cur["short_seen"] = True
            cur["length"] += n
            cur["n_count"] += seq.count("N") + seq.count("n")
            cur["softmasked"] += sum(seq.count(b) for b in "acgtn")
            fout.write(seq + "\n")
            offset += n + 1
        close(cur)

    with open(dst_fa + ".fai", "w") as f:
        f.write("\n".join(fai) + "\n")
    missing = keep - {s["chrom"] for s in stats}
    if missing:
        raise ValueError(f"Chybí chromozomy: {sorted(missing)}")
    return stats

## 3. Stažení a kontrola MD5

In [ ]:
WORK_DIR = str(P.scratch / "genome")     # v Colabu rychlý lokální disk, jinde = cílová složka
OUT_DIR = str(P.genome_dir)
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

check_dns(MIRRORS)
gz_path = os.path.join(WORK_DIR, FA_GZ)
BASE_URL = with_mirrors(lambda base: download(f"{base}/{FA_GZ}", gz_path), MIRRORS)

md5_expected = expected_md5(f"{BASE_URL}/md5sum.txt", FA_GZ)
md5_actual = md5sum(gz_path)
print("očekávané MD5:", md5_expected)
print("spočítané MD5:", md5_actual)
assert md5_actual == md5_expected, "MD5 nesedí, soubor smaž a stáhni znovu"

## 4. Filtrování chromozomů, index a statistiky

In [ ]:
import pandas as pd

fa_path = os.path.join(WORK_DIR, OUT_FA)
t0 = time.time()
stats = filter_fasta(gz_path, fa_path, KEEP)
print(f"Hotovo za {time.time() - t0:.0f} s")

df = pd.DataFrame(stats)
df["n_frac"] = df["n_count"] / df["length"]
df["softmasked_frac"] = df["softmasked"] / df["length"]
df = df.set_index("chrom").loc[KEEP]
print(f"Celkem: {df['length'].sum():,} bp, N: {df['n_count'].sum():,}, "
      f"soft-masked: {df['softmasked'].sum() / df['length'].sum():.1%}")
df

## 5. Uložení

In [ ]:
meta = {
    "source": f"{BASE_URL}/{FA_GZ}",
    "assembly": "GRCh38 (UCSC hg38 analysis set, no alt contigs)",
    "md5_fa_gz": md5_actual,
    "downloaded_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "chromosomes": KEEP,
    "total_bp": int(df["length"].sum()),
    "softmasking": "lowercase = RepeatMasker + Tandem Repeats Finder",
}
with open(os.path.join(WORK_DIR, "metadata.json"), "w") as f:
    json.dump(meta, f, indent=2)
df.to_csv(os.path.join(WORK_DIR, "chrom_stats.csv"))

if os.path.abspath(WORK_DIR) != os.path.abspath(OUT_DIR):
    for name in [FA_GZ, OUT_FA, OUT_FA + ".fai", "metadata.json", "chrom_stats.csv"]:
        src = os.path.join(WORK_DIR, name)
        print(f"Kopíruji {name} ({os.path.getsize(src) / 1e9:.2f} GB)...")
        shutil.copy(src, os.path.join(OUT_DIR, name))
print(sorted(os.listdir(OUT_DIR)))

## 6. Kontrola: čtení přes pyfaidx

In [ ]:
from pyfaidx import Fasta

genome = Fasta(str(P.fasta_stored))   # použije uložený .fai
print(list(genome.keys()))
print(len(genome["chr1"]), "bp na chr1")
print(genome["chr1"][1_000_000:1_000_120])